# 01 · Qué es un LLM

**Python que piensa: agentes de analítica con LangChain y LangGraph**
Sociedad Ecuatoriana de Estadística · Semana de Python

---

1. **Qué es un LLM** y cómo se le habla desde Python, con tres proveedores.
2. **Por qué LangChain**: la misma llamada sirve para todos.
3. **Qué no puede hacer un LLM**: no sabe qué día es ni conoce nuestros datos.

> **Sobre los datos.** Reclamos médicos **sintéticos** del CMS de Estados
> Unidos. Sirven para construir herramientas, no para inferir sobre la población
> real de Medicare.

## Legibilidad para proyección

In [1]:
from IPython.display import HTML, display

# Agranda la letra de las salidas para que se lea desde el fondo de la sala.
display(HTML("""
<style>
  .jp-RenderedText pre, .jp-OutputArea-output pre, div.output_area pre {
      font-size: 15px !important; line-height: 1.5 !important; }
  .jp-RenderedHTMLCommon table { font-size: 15px !important; }
  .jp-InputArea-editor, .CodeMirror { font-size: 15px !important; }
</style>
"""))
print("Tipografía ajustada para proyección.")

Tipografía ajustada para proyección.


## Dónde estamos

In [2]:
from pathlib import Path

# RAIZ = la carpeta del repositorio (la que contiene datos/). Se busca hacia
# arriba desde donde se abrió el notebook, así corre igual en otra máquina.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos").is_dir()), None)
BASE = RAIZ / "datos" / "reclamos.db"     # la base SQLite que armó 00_setup

print(f"Raíz del proyecto : {RAIZ}")
print(f"Base de datos     : {BASE}  {'(existe)' if BASE.exists() else '(falta: corre 00_setup.ipynb)'}")

Raíz del proyecto : C:\Users\ricar\Documents\SEE\Charla2026Sep\PythonWeekend
Base de datos     : C:\Users\ricar\Documents\SEE\Charla2026Sep\PythonWeekend\datos\reclamos.db  (existe)


---

# Parte 1 · Qué es un LLM

Un **LLM** (*large language model*) es un programa que **predice el siguiente
pedazo de texto**. No busca en internet ni consulta bases: completa texto. Vive
en un servidor del proveedor y se le habla por internet.

> **Pausa: las consolas.** En el navegador, el mismo patrón tres veces: una
> cuenta, una **key** y un **modelo**.
>
> - **Azure AI Foundry** ([ai.azure.com](https://ai.azure.com)): catálogo →
>   modelo → **Deploy**; copiar endpoint y key.
> - **OpenAI** ([platform.openai.com](https://platform.openai.com)): API keys →
>   crear key.
> - **Claude** ([console.anthropic.com](https://console.anthropic.com)): API
>   keys → crear key.

## Las credenciales

In [3]:
import os
from dotenv import load_dotenv

# El archivo .env guarda las credenciales. NUNCA se sube al repositorio.
# load_dotenv lo lee y deja cada variable disponible en os.environ.
load_dotenv(RAIZ / ".env")

variables = [
    "AZURE_OPENAI_BASE_URL",   # Foundry: el endpoint, la dirección web donde está el modelo
    "AZURE_OPENAI_API_KEY",    # Foundry: la key, la contraseña que dice quién llama y a quién cobrar
    "AZURE_OPENAI_MODELO",     # Foundry: el deployment, el modelo que desplegamos
    "OPENAI_API_KEY",          # OpenAI directo: su key
    "OPENAI_MODELO",           # OpenAI directo: qué modelo usar
    "ANTHROPIC_API_KEY",       # Claude: su key
    "ANTHROPIC_MODELO",        # Claude: qué modelo usar
]
for variable in variables:
    print(f"{variable:<24} {'definida' if os.getenv(variable) else 'FALTA'}")   # sin mostrar el valor

AZURE_OPENAI_BASE_URL    definida
AZURE_OPENAI_API_KEY     definida
AZURE_OPENAI_MODELO      definida
OPENAI_API_KEY           definida
OPENAI_MODELO            definida
ANTHROPIC_API_KEY        definida
ANTHROPIC_MODELO         definida


## Lo que cuesta cada pregunta

Cada llamada al modelo **cuesta dinero**, y se cobra por token. En este notebook
cada respuesta muestra **qué modelo respondió, cuántos tokens usó y cuánto
costó**, más el acumulado.

Precios oficiales por millón de tokens, consultados el 2026-10-07:

| Modelo | Entrada | Entrada en caché | Salida | Fuente |
|---|---:|---:|---:|---|
| gpt-4.1 (Foundry y OpenAI) | 2,00 | 0,50 | 8,00 | [OpenAI](https://developers.openai.com/api/docs/models/gpt-4.1), [Azure](https://azure.microsoft.com/es-es/pricing/details/azure-openai) (Global Standard) |
| claude-sonnet-5-5 | 2,00 | 0,20 | 10,00 | [Anthropic](https://platform.claude.com/docs/en/about-claude/pricing) |

In [4]:
# USD por MILLÓN de tokens. Si un proveedor cambia sus precios, se cambia aquí.
PRECIOS = {
    "gpt-4.1":           {"entrada": 2.00, "cache_lectura": 0.50, "cache_escritura": 2.00, "salida": 8.00},
    "claude-sonnet-5-5": {"entrada": 2.00, "cache_lectura": 0.20, "cache_escritura": 2.50, "salida": 10.00},
}
gasto_total = 0.0    # lo que llevamos gastado en este notebook, en USD

# Esta función se usa después de CADA llamada. No toca el modelo: solo lee la
# cuenta de tokens de la respuesta y la multiplica por el precio.
def costo(respuesta, proveedor):
    global gasto_total
    modelo_usado = respuesta.response_metadata["model_name"]    # el modelo que respondió (ej. gpt-4.1-2025-04-14)
    precio = next(p for nombre, p in PRECIOS.items() if modelo_usado.startswith(nombre))

    uso = respuesta.usage_metadata                              # la cuenta de tokens de esta respuesta
    leidos_de_cache   = uso["input_token_details"].get("cache_read", 0)       # entrada que ya estaba en caché
    escritos_en_cache = (uso["input_token_details"].get("cache_creation", 0)  # entrada que se guardó en caché
                         or uso["input_token_details"].get("ephemeral_5m_input_tokens", 0))   # (Claude la informa aquí)
    entrada_normal    = uso["input_tokens"] - leidos_de_cache - escritos_en_cache

    usd = (entrada_normal    * precio["entrada"]
         + leidos_de_cache   * precio["cache_lectura"]
         + escritos_en_cache * precio["cache_escritura"]
         + uso["output_tokens"] * precio["salida"]) / 1_000_000
    gasto_total += usd

    print(f"   [{proveedor} · {modelo_usado}] {uso['input_tokens']} tokens de entrada, "
          f"{uso['output_tokens']} de salida -> USD {usd:.6f} | acumulado USD {gasto_total:.6f}")

## Ejemplo 1 · La misma pregunta, de cuatro formas

Una pregunta sencilla. Primero con la biblioteca **original** del proveedor;
después con **LangChain**, contra tres proveedores distintos.

In [5]:
pregunta = "Describe la primera ley de Newton en cinco palabras."

### 1a · Con la biblioteca original de OpenAI, contra Foundry

In [6]:
from openai import OpenAI

# El cliente es el objeto que se conecta al modelo. La biblioteca openai sirve
# para Foundry porque su endpoint habla el mismo idioma que la API de OpenAI.
cliente = OpenAI(
    base_url=os.environ["AZURE_OPENAI_BASE_URL"],   # a dónde llamar
    api_key=os.environ["AZURE_OPENAI_API_KEY"],     # quién llama
)

respuesta = cliente.chat.completions.create(
    model=os.environ["AZURE_OPENAI_MODELO"],     # a qué modelo le preguntamos
    messages=[                                   # la conversación: una lista de mensajes
        {"role": "user",                         # quién habla: la persona que pregunta
         "content": pregunta},                   # qué dice
    ],
)

# La respuesta trae una lista "choices" (se pueden pedir varias alternativas).
# Pedimos una sola, así que leemos la primera: choices[0].
print("OpenAI SDK + Foundry:", respuesta.choices[0].message.content)

# El costo, calculado a mano: tokens x precio por millón.
uso = respuesta.usage
usd = (uso.prompt_tokens * PRECIOS["gpt-4.1"]["entrada"]
     + uso.completion_tokens * PRECIOS["gpt-4.1"]["salida"]) / 1_000_000
gasto_total += usd
print(f"   [Foundry · {respuesta.model}] {uso.prompt_tokens} tokens de entrada, "
      f"{uso.completion_tokens} de salida -> USD {usd:.6f} | acumulado USD {gasto_total:.6f}")

OpenAI SDK + Foundry: Todo cuerpo persevera en reposo.
   [Foundry · gpt-4.1-2025-04-14] 17 tokens de entrada, 8 de salida -> USD 0.000098 | acumulado USD 0.000098


### 1b · Con LangChain, contra Foundry

In [7]:
from langchain_openai import ChatOpenAI

# LangChain envuelve al proveedor en un objeto "modelo" con una interfaz común.
foundry = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_BASE_URL"],   # a dónde llamar
    api_key=os.environ["AZURE_OPENAI_API_KEY"],     # quién llama
    model=os.environ["AZURE_OPENAI_MODELO"],        # qué modelo
)

respuesta = foundry.invoke(pregunta)   # invoke: preguntar. Acepta un texto o una lista de mensajes
print("LangChain + Foundry :", respuesta.text)   # .text: solo el texto de la respuesta
costo(respuesta, "Foundry")

LangChain + Foundry : Inercia: reposo o movimiento constante.
   [Foundry · gpt-4.1-2025-04-14] 17 tokens de entrada, 10 de salida -> USD 0.000114 | acumulado USD 0.000212


### 1c · Con LangChain, contra OpenAI directo

In [8]:
# Mismo tipo de objeto, sin endpoint: va a la API pública de OpenAI.
# La key la toma sola de la variable OPENAI_API_KEY.
openai_directo = ChatOpenAI(model=os.environ["OPENAI_MODELO"])

respuesta = openai_directo.invoke(pregunta)
print("LangChain + OpenAI  :", respuesta.text)
costo(respuesta, "OpenAI")

LangChain + OpenAI  : Todo cuerpo permanece en reposo.
   [OpenAI · gpt-4.1-2025-04-14] 17 tokens de entrada, 7 de salida -> USD 0.000090 | acumulado USD 0.000302


### 1d · Con LangChain, contra Claude

In [9]:
from langchain_anthropic import ChatAnthropic

# Otro proveedor, otra clase. La key la toma sola de ANTHROPIC_API_KEY.
claude = ChatAnthropic(model=os.environ["ANTHROPIC_MODELO"])

respuesta = claude.invoke(pregunta)
print("LangChain + Claude  :", respuesta.text)
costo(respuesta, "Claude")

LangChain + Claude  : **Todo cuerpo persiste en su estado.**

(Cinco palabras: Todo · cuerpo · persiste · en · su · estado... en realidad son seis, así que aquí va una versión exacta:)

**Todo objeto mantiene su movimiento.**

Cinco palabras: Todo · objeto · mantiene · su · movimiento.

Otra opción: **La inercia mantiene el movimiento.**
   [Claude · claude-sonnet-5-5] 28 tokens de entrada, 144 de salida -> USD 0.001496 | acumulado USD 0.001798


### Lo que cambia y lo que no

Con LangChain, **la llamada es la misma para los tres**: `modelo.invoke(...)` y
`.text`. Lo único que cambia es la línea donde se crea el modelo. Con las
bibliotecas originales, cada proveedor tiene su propia forma de llamar y de
leer la respuesta.

In [10]:
import time

modelos = {"Foundry": foundry, "OpenAI": openai_directo, "Claude": claude}

for nombre, modelo in modelos.items():
    inicio = time.perf_counter()
    respuesta = modelo.invoke(pregunta)          # la MISMA línea para los tres
    segundos = time.perf_counter() - inicio

    print(f"{nombre} ({segundos:.1f} s): {respuesta.text.strip().replace(chr(10), ' ')[:70]}")
    costo(respuesta, nombre)                     # usage_metadata viene en el mismo formato para todos

# Claude Sonnet razona antes de responder: esos tokens de razonamiento también
# se cuentan (y se cobran) como tokens de salida.

Foundry (1.3 s): Cuerpo en reposo permanece quieto.
   [Foundry · gpt-4.1-2025-04-14] 17 tokens de entrada, 9 de salida -> USD 0.000106 | acumulado USD 0.001904


OpenAI (0.7 s): Cuerpo en reposo permanece igual.
   [OpenAI · gpt-4.1-2025-04-14] 17 tokens de entrada, 8 de salida -> USD 0.000098 | acumulado USD 0.002002


Claude (1.6 s): **Todo cuerpo persiste en su estado.**  (Son cinco palabras: la inerci
   [Claude · claude-sonnet-5-5] 28 tokens de entrada, 80 de salida -> USD 0.000856 | acumulado USD 0.002858


---

## De aquí en adelante: LangChain

Todo el resto de la charla usa LangChain. Elegimos el modelo de **Foundry**;
cambiarlo por cualquiera de los otros dos es cambiar esta línea.

In [11]:
modelo = foundry         # o: openai_directo, o: claude
PROVEEDOR = "Foundry"    # para el registro de costo
print("Modelo de trabajo:", PROVEEDOR, "·", modelo.model_name)

Modelo de trabajo: Foundry · gpt-4.1


## Ejemplo 2 · El mensaje `system` cambia cómo responde

In [12]:
import textwrap   # corta el texto largo en líneas de 90 caracteres, para leerlo en pantalla
from langchain_core.messages import SystemMessage, HumanMessage

respuesta = modelo.invoke([
    # SystemMessage: la instrucción de fondo. No es la pregunta: le dice al
    # modelo quién debe ser y cómo responder durante toda la conversación.
    SystemMessage("Eres un profesor de primaria. Explica con un ejemplo de la vida diaria, en dos frases."),
    # HumanMessage: lo que pregunta la persona.
    HumanMessage("¿Qué dice la primera ley de Newton?"),
])

print(textwrap.fill(respuesta.text, 90))
costo(respuesta, PROVEEDOR)

La primera ley de Newton dice que un objeto quieto sigue quieto y uno en movimiento sigue
moviéndose, a menos que algo lo detenga o lo empuje. Por ejemplo, si empujas un carrito y
nadie lo detiene, seguirá rodando, pero si choca con una pared, se detendrá.
   [Foundry · gpt-4.1-2025-04-14] 40 tokens de entrada, 66 de salida -> USD 0.000608 | acumulado USD 0.003466


## Ejemplo 3 · La temperatura

In [13]:
pedido = "Cuéntame un chiste corto sobre programadores."

# temperature: cuánto se arriesga el modelo al elegir la siguiente palabra.
#   0 -> elige casi siempre la más probable: repite la misma respuesta.
#   1 -> se permite palabras menos probables: cada respuesta puede cambiar.
# Se pasa directo en invoke. (Claude Sonnet 5.5 no la acepta: los modelos que
# razonan antes de responder controlan su variación de otra forma.)
for temperatura in [0, 1]:
    print(f"--- temperatura {temperatura} ---")
    for intento in [1, 2, 3]:
        respuesta = modelo.invoke(pedido, temperature=temperatura)
        print(f"Intento {intento}:", respuesta.text.replace("\n", " "))
        costo(respuesta, PROVEEDOR)
    print()

# Para analítica se usa temperatura 0: misma pregunta, mismo SQL, sin creatividad.

--- temperatura 0 ---


Intento 1: ¿Por qué los programadores confunden Halloween con Navidad?  Porque OCT 31 = DEC 25.
   [Foundry · gpt-4.1-2025-04-14] 18 tokens de entrada, 20 de salida -> USD 0.000196 | acumulado USD 0.003662


Intento 2: ¿Por qué los programadores confunden Halloween con Navidad?  Porque OCT 31 = DEC 25.
   [Foundry · gpt-4.1-2025-04-14] 18 tokens de entrada, 20 de salida -> USD 0.000196 | acumulado USD 0.003858


Intento 3: ¿Por qué los programadores confunden Halloween con Navidad?  Porque OCT 31 = DEC 25.
   [Foundry · gpt-4.1-2025-04-14] 18 tokens de entrada, 20 de salida -> USD 0.000196 | acumulado USD 0.004054

--- temperatura 1 ---


Intento 1: ¿Por qué los programadores confunden Halloween con Navidad?  ¡Porque OCT 31 es igual a DEC 25!
   [Foundry · gpt-4.1-2025-04-14] 18 tokens de entrada, 23 de salida -> USD 0.000220 | acumulado USD 0.004274


Intento 2: ¿Por qué los programadores confunden Halloween con Navidad?  Porque OCT 31 es igual a DEC 25.
   [Foundry · gpt-4.1-2025-04-14] 18 tokens de entrada, 22 de salida -> USD 0.000212 | acumulado USD 0.004486


Intento 3: ¿Cuál es el colmo de un programador?  ¡Tener problemas de *cache* en la memoria!
   [Foundry · gpt-4.1-2025-04-14] 18 tokens de entrada, 23 de salida -> USD 0.000220 | acumulado USD 0.004706



## Ejemplo 4 · Los tokens

In [14]:
# Un token es un pedazo de palabra (unos 4 caracteres en promedio). El modelo lee
# y escribe en tokens, y el proveedor COBRA POR TOKEN: los de entrada (la
# pregunta) y los de salida (la respuesta) se cobran por separado.
respuesta = modelo.invoke("Explica qué es la fotosíntesis.",
                          max_tokens=15)     # tope de tokens de la respuesta: se corta ahí

print("Respuesta        :", respuesta.text)
print("Tokens de entrada:", respuesta.usage_metadata["input_tokens"])
print("Tokens de salida :", respuesta.usage_metadata["output_tokens"])
print("Por qué terminó  :", respuesta.response_metadata["finish_reason"])   # length = se acabó el tope
costo(respuesta, PROVEEDOR)

Respuesta        : La **fotosíntesis** es un proceso mediante el cual las
Tokens de entrada: 17
Tokens de salida : 15
Por qué terminó  : length
   [Foundry · gpt-4.1-2025-04-14] 17 tokens de entrada, 15 de salida -> USD 0.000154 | acumulado USD 0.004860


---

# Parte 2 · Lo que un LLM no puede hacer

El modelo solo sabe lo que leyó al entrenarse.

## No sabe qué día es

In [15]:
from datetime import date

# Su conocimiento se congeló al terminar el entrenamiento, y no tiene reloj.
respuesta = modelo.invoke("¿Qué fecha es hoy? Responde solo con la fecha.")

print("El modelo dice:", respuesta.text)
print("Hoy es        :", date.today().isoformat())
costo(respuesta, PROVEEDOR)

El modelo dice: 10 de junio de 2024
Hoy es        : 2026-10-07
   [Foundry · gpt-4.1-2025-04-14] 19 tokens de entrada, 7 de salida -> USD 0.000094 | acumulado USD 0.004954


## No conoce nuestros datos

In [16]:
import sqlite3

pregunta = ("En mi base de reclamos médicos, ¿cuánto se pagó en hospitalización "
            "en 2009? Responde en máximo dos frases.")

# El modelo nunca vio nuestra base: si no le damos nada, no puede saber nada de ella.
respuesta = modelo.invoke(pregunta, temperature=0)

# La cifra real, consultada directamente en la base.
con = sqlite3.connect(BASE)
real = con.execute(
    "SELECT sum(monto_pagado) FROM sabana WHERE ambito = 'Hospitalario' AND anio = 2009"
).fetchone()[0]
con.close()

print("Pregunta:")
print(textwrap.fill(pregunta, 90))
print()
print("El modelo:")
print(textwrap.fill(respuesta.text, 90))
print()
print(f"La base: {real:,.2f}")
costo(respuesta, PROVEEDOR)

Pregunta:
En mi base de reclamos médicos, ¿cuánto se pagó en hospitalización en 2009? Responde en
máximo dos frases.

El modelo:
Para responder a tu pregunta, necesito acceso a los datos de tu base de reclamos médicos.
Por favor, proporciona la información o el archivo correspondiente para que pueda analizar
cuánto se pagó en hospitalización en 2009.

La base: 245,512,790.00
   [Foundry · gpt-4.1-2025-04-14] 36 tokens de entrada, 45 de salida -> USD 0.000432 | acumulado USD 0.005386


## Lo que gastamos en este notebook

In [17]:
print(f"Gasto total de este notebook: USD {gasto_total:.6f}")
print(f"Correrlo 100 veces costaría:  USD {gasto_total * 100:.4f}")

Gasto total de este notebook: USD 0.005386
Correrlo 100 veces costaría:  USD 0.5386


---

## Cierre

> **Un LLM predice texto. No sabe qué día es ni conoce nuestros datos.**

- Se le habla con un modelo, una lista de mensajes y unos pocos parámetros.
- Con LangChain, la llamada es la misma para cualquier proveedor.
- Para que responda sobre nuestra base hay que **darle el contexto** (qué tabla,
  qué columnas, qué valores) y **una herramienta** para consultarla. Eso es el
  notebook 02.

### Decisión de arquitectura · ¿de dónde sale el modelo?

| Camino | Cuándo conviene | Qué se gana | Qué se paga |
|---|---|---|---|
| **Azure AI Foundry** (nube corporativa) | Banca, seguros, sector público | El dato no sale de tu nube, facturación corporativa, modelos de varios proveedores | Más pasos para empezar |
| **API directa del proveedor** (OpenAI, Anthropic) | Prototipos, equipos pequeños | Lo más simple; los modelos nuevos llegan primero | El dato viaja al proveedor |
| **Suscripción de producto** (el chat web) | Uso personal | Nada que construir | No sirve para construir: no hay API |

Con LangChain, esta decisión **no queda amarrada al código**: cambiar de camino
es cambiar la línea que crea el modelo.

### Tres preguntas de entrevista

1. ¿Qué gana un equipo al usar LangChain en vez de la biblioteca original del
   proveedor, y qué pierde?
2. ¿Por qué se usa temperatura 0 para generar SQL, y por qué eso no garantiza
   que la respuesta sea correcta?
3. Un modelo que razona antes de responder gasta más tokens de salida. ¿Cuándo
   vale la pena pagar ese costo y cuándo no?